About this File:

This Jupyter Notebook is meant to clean and analyze a sample of the Ubuntu Dialogue Corpus V1.0. The Ubuntu Dialogue Corpus is a large dataset of text-based technical support conversations extracted from Ubuntu chat logs on the Freenode IRC network, containing nearly 1 million multi-turn, two-person dialogues used to train and evaluate conversational agents and neural dialogue model

The sample of the dataset used for this project specifically is sourced from Kaggle.

---
Kaggle Link: https://www.kaggle.com/datasets/rtatman/ubuntu-dialogue-corpus

Full Ubuntu Dialogue Corpus Version 1.0 Link: https://dataset.cs.mcgill.ca/ubuntu-corpus-1.0/

                                                                       
Ubuntu Version 2.0 Github Link: https://github.com/rkadlec/ubuntu-ranking-dataset-creator


---
Step 0. 
    
- What are we solving?
- What does success look like?
- What data do we need?

Before looking at the data itself, my goal is to understand the communication between users on this live chat forum, who are assisting one another with technical support questions. Because I was unfamiliar with Ubuntu prior to discovering this dataset, I have done some research to understand the site, and plan to do the same for technical jargon contained in conversations, so that I can build out a supplementary lexicon to improve NLP analysis, such as for correcting misspellings via fuzzy-matching.

Problem being solved > Help Ubuntu recognize and predict what topics its users are discussing on this forum, what frustrations or other primary feelings they have, and how user experiences evolve over time — particularly in relation to Ubuntu's release calendar. The goal is to move Ubuntu from reactive (responding thread by thread) to proactive: knowing which issues cause outsized frustration, when support demand will spike, and whether friction comes from new users onboarding or experienced users hitting recurring bugs.

Success looks like > Ubuntu anticipates user behavior and produces centralized forums for repeat issues, specifically by being able to:
- Identify topics/packages with disproportionate frustration per mention — pointing to where fixes or docs matter most.
- Anticipate support-volume spikes tied to Ubuntu's release calendar.
- Distinguish onboarding friction (low-frequency users abandoning threads) from systemic pain points (recurring issues even high-frequency users hit).
- Catch emerging issues early — flag a topic as a likely recurring problem based on how fast negative sentiment around it accelerates.

What data is needed > The core corpus: user IDs, message text, timestamps, and conversation grouping, to reconstruct threads rather than isolated lines. +Plus:

- A researched Ubuntu/Linux jargon lexicon, for fuzzy-matching and misspelling correction that generic NLP tools would flag as typos.
- A reference table of Ubuntu release dates, for correlating chat volume/sentiment to upgrade cycles.
- A sentiment/frustration measure, derived from message text since it's not in the raw data.
- User-level activity counts, to separate power users from occasional posters.
- Optional if time allows: rolling topic-cluster detection over time, to support early-warning flagging

---

Step 1: Data Validation & Gap Analysis
- Review data in Excel/Viewer
- Identify missing columns
- Document expected vs actual

Step 2: Import Libraries
- Import standard libraries
- Set display options
- Configure warnings

Step 3: Load Data
- Read CSV/Excel/JSON
- Handle encoding issues
- Save as df

Step 4: Initial Inspection 🔍
- df.head() — what's in there?
- df.info() — what types?
- df.describe() — basic stats?
- df.shape — how big?

>(Performed together, out of sequence, as dataset was loaded and inspected within this notebook, due to its size.)

---

In [ ]:
#Import Libraries
import numpy as np
import pandas as pd

In [2]:
#Data Validation and Gap Analysis
pd.set_option('display.max_colwidth', None)  # Show full content of columns
path = "../_corpus_data/dialogueText_196.csv" #I used the 0.9gb file, however, you can use any sample size or all of the data if it's in the same format.
df_sample = pd.read_csv(path, nrows = 20)
df_sample

,folder,dialogueID,date,from,to,text
0,301,1.tsv,2004-11-23T11:49:00.000Z,stuNNed,NaN,any ideas why java plugin takes so long to load?
1,301,1.tsv,2004-11-23T11:49:00.000Z,crimsun,stuNNed,java 1.4?
2,301,1.tsv,2004-11-23T11:49:00.000Z,stuNNed,crimsun,yes
3,301,1.tsv,2004-11-23T11:49:00.000Z,crimsun,stuNNed,java 1.5 loads _much_ faster
4,301,1.tsv,2004-11-23T11:50:00.000Z,stuNNed,crimsun,noneus: how can i get 1.5 is there a .deb somewhere?
5,301,1.tsv,2004-11-23T11:50:00.000Z,crimsun,stuNNed,not yet.
6,301,1.tsv,2004-11-23T11:50:00.000Z,stuNNed,crimsun,noneus: is this blackdown or sun?
7,301,1.tsv,2004-11-23T11:50:00.000Z,stuNNed,crimsun,did you install just the jre?
8,301,1.tsv,2004-11-23T11:51:00.000Z,crimsun,stuNNed,I use IBM's 1.4.2
9,301,1.tsv,2004-11-23T11:51:00.000Z,crimsun,stuNNed,"(jdk, because I do globus development)"


In [3]:
df_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   folder      20 non-null     int64
 1   dialogueID  20 non-null     str  
 2   date        20 non-null     str  
 3   from        20 non-null     str  
 4   to          19 non-null     str  
 5   text        20 non-null     str  
dtypes: int64(1), str(5)
memory usage: 2.5 KB


---
Notes: 
- I have not yet loaded the full dataset into the notebook, so describe(), and shape(), have not yet been run.
- On full dataset, I should count unique for folder column to determine its relevance or necessity. If all chats are from the same folder, I will delete the column. If relevant, I will downcast to a much smaller uint
- Similarly, I will remove the .tsv suffix from dialogueID column and convert and downcast the column to from string to a much smaller uint
- date column needs to be converted from str to a datetime format and stripped of seconds and milliseconds which do not appear to be stored (will confirm). UTC timezone for all rows
- from and to do not appear to have any issues.
- Will confirm: The first message from the original poster (OP) will always be to "NaN" before another user picks up and replies, starting a thread. 
- Creating a column with ordinal turn ID, grouped by dialogue ID and beginning at the row where the OP messages to "NaN" until the dialogue ID changes will help track patterns across chat turns
- Potentially adding a boolean column from_OP to track less granular sentiment patterns
- text column contains raw conversational messages. Sentences in the sample show topics like Java versions, IBM, jre packages, Globus, loading speed, and URL links. Typos noticed are underscores within sentences, jdk instead of idk. Casual conversation results in inconsistent casing, grammar, and frequent misspellings or typos. 
-Other columns to add would be the sentiment/emotions expressed in the text after cleaning.

Next is loading the dataset while saving on RAM by downcasting where applicable. Converting the "From" and "To" usernames to category will save on RAM due to assumed massive repetition of users. 

In [4]:
#Load full dataset with efficiency 
dtype_map = {
    'folder': 'uint16',   # downcast later depending on amount of folders in dataset
    'from': 'category',   # usernames repeat many times; category saves massive RAM here
    'to': 'category',
}
df = pd.read_csv(
    path,
    dtype=dtype_map,
    parse_dates=['date'],
    date_format='%Y-%m-%dT%H:%M:%S.%fZ',  # explicit format
)
df['dialogueID'] = pd.to_numeric(
    df['dialogueID'].str.replace('.tsv', '', regex=False),
    downcast='integer'
)

In [5]:
#Check whether second and millisecond is relevant
print(df['date'].dt.second.nunique())
print(df['date'].dt.microsecond.nunique())

1
1


In [6]:
df['date'] = df['date'].dt.floor('min') #Remove milliseconds

In [7]:
df.shape
#df.describe() doesn't make sense with this data

(9212877, 6)

In [8]:
df['folder'].nunique() #count how many unique values there are for the "folder" column

173

In [9]:
#check whether dialogue IDs repeat across different folders
print(df['dialogueID'].nunique())
print(df.groupby(['folder', 'dialogueID']).ngroups)

346108
1008391


In [10]:
df.groupby(['folder', 'dialogueID'])['from'].nunique().value_counts() #Check how many of the conversations have two distinct speakers based on "from"

from
2    999648
1      8743
Name: count, dtype: int64

In [11]:
df.groupby(['folder', 'dialogueID'])['to'].nunique().value_counts() #Check how many of the conversations have two distinct speakers based on "to"

to
2    522604
1    485776
0        11
Name: count, dtype: int64

In [12]:
# find a dialogueID that shows up in more than one folder to demonstrate that dialogueIDs are not unique except within the folder itself
multi_folder_ids = df.groupby('dialogueID')['folder'].nunique()
candidate = multi_folder_ids[multi_folder_ids > 1].index[5]

check = df[df['dialogueID'] == candidate].sort_values('date')
check[['folder', 'date', 'from', 'to', 'text']]

,folder,date,from,to,text
6607177,10,2004-09-24 11:09:00,jamesh,NaN,xdpyinfo is probably the command you want
6607178,10,2004-09-24 11:09:00,fabbione,jamesh,the output is correct
6607180,10,2004-09-24 11:11:00,jamesh,fabbione,apparently Keith reckons the new X extensions will get rid of the need for Xinerama
6607179,10,2004-09-24 11:11:00,fabbione,jamesh,yeah..
6607181,10,2004-09-24 11:11:00,jamesh,fabbione,since you'd be able to composite windows from one screen to the other
...,...,...,...,...,...
4273175,40,2012-11-28 14:27:00,Erin,RJ45-Q,is smbd running ?
4273177,40,2012-11-28 14:28:00,Erin,RJ45-Q,ps -ef | grep smbd
4273178,40,2012-11-28 14:28:00,Erin,RJ45-Q,sudo service smbd start or do restart
4273176,40,2012-11-28 14:28:00,RJ45-Q,Erin,should be


Above, I have loaded the full dataset with tentatively best practices for efficiency, via downcasting where applicable, converting usernames to categories, and removing excess precision from datetime format. Further, I now understand that the folder column is unique, and dialogue IDs are unique only within folders, not between. 

---

Step 5: Clean Data 🧹

5a: Column Names
- strip whitespace
- lower case
- replace spaces with _

5b: Row Data
- Check/remove duplicates
- Fix data types  >I did this earlier because the data set is very large and necessitated prioritizing for efficiency.
- Handle missing values
- Sort data if needed

In [13]:
#Column Name Fixes
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df = df.rename(columns={'dialogueid': 'dialogue_id'})

In [14]:
df.columns #check

Index(['folder', 'dialogue_id', 'date', 'from', 'to', 'text'], dtype='str')

In [15]:
#Duplicates
df.duplicated().sum() #check how many rows are duplicates, excluding original rows

np.int64(25161)

In [16]:
df.duplicated(keep=False).sum() #includes original rows and their duplicates

np.int64(44131)

In [17]:
df[df.duplicated(keep=False)]['folder'].value_counts().head(10) #check duplicates per folder

folder
11    12439
5      6554
10     4356
12     4208
3      3270
13     2257
14     1784
15     1494
16     1104
17      999
Name: count, dtype: int64

In [18]:
#Check how many times a message is being copied based on counting the number of duplicates per original row
dup_group_sizes = df.groupby(list(df.columns)).size()
dup_group_sizes[dup_group_sizes > 1].value_counts()

2     2862
3       79
5       13
4       12
7        3
10       2
8        2
6        1
12       1
9        1
13       1
19       1
Name: count, dtype: int64

In [19]:
dup_group_sizes[dup_group_sizes == 19] #examine the message being duplicated 19 times

folder  dialogue_id  date                 from     to       text                  
29      3688         2006-03-01 07:59:00  damian_  damian_  wats that mean in wine    19
dtype: int64

In [20]:
df.duplicated(subset=['date','from', 'to', 'text'], keep=False).sum() #examine if duplicates exist across folders, despite different dialogue_id

np.int64(1116311)

In [21]:
df[df.duplicated(subset=['date','from', 'to', 'text'], keep=False)].sort_values(['from', 'to', 'text']).head(4) #examine sample

,folder,dialogue_id,date,from,to,text
4730913,12,35688,2006-12-02 08:12:00,A3n,NaN,and for ndiswrapper i need 'make install'
7703124,11,66043,2006-12-02 08:12:00,A3n,NaN,and for ndiswrapper i need 'make install'
3622395,5,196907,2006-12-06 11:01:00,A3n,NaN,how do i start wlan0 ?
7885291,11,54523,2006-12-06 11:01:00,A3n,NaN,how do i start wlan0 ?


In [22]:
df = df.drop_duplicates(subset=['date', 'from', 'text'], keep='first').reset_index(drop=True) #Drop duplicates across folders

In [23]:
df.shape #check

(8590361, 6)

In [24]:
df.duplicated(subset=['date', 'from', 'text']).sum() #check

np.int64(0)

In [25]:
df.isna().sum() #examine how many values are NaN per column

folder               0
dialogue_id          0
date                 0
from               197
to             2687247
text               433
dtype: int64

In [26]:
df = df.dropna(subset=['text']).reset_index(drop=True) #delete all rows with no text

In [27]:
df.shape

(8589928, 6)

In [28]:
df.isna().sum()

folder               0
dialogue_id          0
date                 0
from               197
to             2686830
text                 0
dtype: int64

In [29]:
df[df['to'].isna()].sample(5)[['folder', 'dialogue_id', 'from', 'text']] #examine NaN in the "to" column

,folder,dialogue_id,from,text
150316,20,11544,Zugot,but breezy is really just hoary right now. not much work has gone into it
3581965,5,184090,refka,Building dependency tree
1130661,3,197388,CopyWriter,hello and good morning all
5942163,10,42937,Shaba1,I could not tell you on linux
6646725,11,22111,soier,in my ubuntu system !


In [30]:
df[df['from'].isna()].sample(5)[['folder', 'dialogue_id', 'to', 'text']] #examine NaN in the "from" column

,folder,dialogue_id,to,text
3522717,5,185144,NaN,how do I unhide it and put it on my desktop?
898261,17,32,jdub,any pointers ? i hear its debian based whats the basic commands n things need to know about ubuntu ?
2348052,14,24634,Ademan,": heh now that I think about it, I'm gonna try calling tech support. I just bought the game like an hour ago, and they did provide a linux client of it. and no overheats... im sure on the background processes, I can watch my cpu usage when not in the game and it stays at like 0 to 2% max"
5180614,15,23773,NaN,cos every packaged version of mplayer is old
5596419,22,9196,NaN,"im getting 510m, soon prolly in another week,"


In [31]:
#handle NAN in "from" and "to" - Claude 
df = df.sort_values(['folder', 'dialogue_id', 'date']).reset_index(drop=True)

def other_of(parts, known):
    rest = parts - {known}
    return next(iter(rest)) if len(rest) == 1 else np.nan

# STEP 1 — fill 'from' nulls via elimination, using ORIGINAL participant sets (pre-fill)
group_sets = df.groupby(['folder', 'dialogue_id'])['from'].apply(lambda s: frozenset(s.dropna()))
orig_participants = df.join(group_sets.rename('_parts'), on=['folder', 'dialogue_id'])['_parts']
from_null = df['from'].isna()
can_infer_from = from_null & (orig_participants.apply(len) == 2) & df['to'].notna()
df.loc[can_infer_from, 'from'] = [
    other_of(p, t) for p, t in zip(orig_participants[can_infer_from], df.loc[can_infer_from, 'to'])
]

# STEP 2 — drop rows where 'from' is still unresolved (no determinable "other user")
df = df[df['from'].notna()].reset_index(drop=True)

# Labels guaranteed not to collide with real values, then register as categories
NO_REPLY_LABEL = '__UNANSWERED__'
BROADCAST_LABEL = '__ALL__'

existing_values = set(df['from'].cat.categories) | set(df['to'].cat.categories)
assert NO_REPLY_LABEL not in existing_values
assert BROADCAST_LABEL not in existing_values

df['to'] = df['to'].cat.add_categories(
    df['from'].cat.categories.difference(df['to'].cat.categories).tolist() + [BROADCAST_LABEL, NO_REPLY_LABEL]
)

# STEP 3 — recompute participants/flags on the now-cleaned 'from' column
group_sets2 = df.groupby(['folder', 'dialogue_id'])['from'].apply(lambda s: frozenset(s.dropna()))
participants2 = df.join(group_sets2.rename('_parts2'), on=['folder', 'dialogue_id'])['_parts2']
is_unanswered = participants2.apply(len) == 1
is_first_msg = df.groupby(['folder', 'dialogue_id']).cumcount() == 0
to_null = df['to'].isna()

df.loc[to_null & is_unanswered, 'to'] = NO_REPLY_LABEL
df.loc[to_null & is_first_msg & ~is_unanswered, 'to'] = BROADCAST_LABEL

remaining = to_null & ~is_first_msg & ~is_unanswered
df.loc[remaining, 'to'] = [
    other_of(p, f) for p, f in zip(participants2[remaining], df.loc[remaining, 'from'])
]

In [32]:
df.isna().sum()

folder         0
dialogue_id    0
date           0
from           0
to             0
text           0
dtype: int64

---
Notes:

When checking duplicates, I began with just searching for exact copies, having the same folder number and dialogue ID, which was 25,161 (for 0.9gb sample dataset). Then, I thought to check if there were duplicates across folders having different dialogue IDs, but having the same date, from, to, and text. This yielded a total of 1,116,311 duplicates! dropped.

Moving onto NaNs:

The 433 NaNs in the text column should have the entire row removed as they are rendered useless. Easy fix.

My hypothesis that missing values (NaN) in the "to" column merely to designate the OP's initial message was oversimplified. NaNs exist in "from" and "to" far too often for username NANs to be written off as conversation starters. That is only one explanation. However, gratefully, much of the time, the other user can be inferred by conversation ID while grouping by folder. 

Method for filling these missing values:
- TO: there are two categories of NA in the "to" column: 1. "Unanswered" (for first messages that never have a followup within that folder and dialogue ID) and 2. "All" for first messages that do have a followup). All other NA in the "to" column should be filled with the other user in the folder's dialogue ID. 
- FROM: they should all be filled with the other user in the folder's dialogue ID. if there is no other user in the dialogue's ID, it's removed altogether, because now its just an unknown sender to anyone /everyone.

>Claude helped big time with writing the code for my logic above

---

Step 6: Derived Data 🛠
- Create new columns
- Calculate new metrics
- Encode categoricals
- Feature engineering

I feel like I should get rid of the folders. make every conversation ID unique. basically, go through each folder and map the first conversation in the first folder to "1" ...then after going through that whole folder of n conversations, the first conversation in the next folder is just n+1, and so on and so forth. That is just so much more intuitive and can plug and play with larger datasets.

this would replace the old dialogue_id, being sure to downcast to the appropriate int size, and there will no longer be a "folder" column

derived columns will be: 
- conversation_id - described above
- turn_count, per conversation, starting at 0 the first or only message in a dialogue_id...int
- is_op boolean...whether the speaker is the OP for less granular analysis
- response_gap_mins - how much time passed between the user's message and the prior message in the conversation, is always N/A for the first message, which should be handled by filtering when turn_count = 0
- user_message_count - how many accumulated messages the user has sent at the explicit point in time that message is being sent. first ever message is always 1
- text_length - character count of a message, spaces inclusive
- word_count - word count of a message
- days_until_release - (after researching past release schedule), time difference between message and upcoming release date, is always N/A for messages prior to the first Ubuntu release. (includes all messages from July 5, 2004 - October 27, 2004)
- days_since_release - (after researching past release schedule), time difference between message and just prior release date

In [33]:
#New Columns
#conversation id
df['conversation_id'] = df.groupby(['folder', 'dialogue_id'], sort=False).ngroup() + 1
df['conversation_id'] = pd.to_numeric(df['conversation_id'], downcast='unsigned')

df = df.drop(columns=['folder', 'dialogue_id'])

In [34]:
# turn_count
df['turn_count'] = df.groupby('conversation_id').cumcount()

In [35]:
# is_op
op_per_conversation = df.groupby('conversation_id')['from'].transform('first')
df['is_op'] = df['from'] == op_per_conversation

In [36]:
# response_gap — in minutes, 0 for the first message of each conversation
df['response_gap_mins'] = df.groupby('conversation_id')['date'].diff().dt.total_seconds() / 60

In [37]:
# user_message_count — cumulative, chronological, 1-indexed
counts = df.sort_values(['from', 'date']).groupby('from').cumcount().add(1)
df['user_message_count'] = counts.reindex(df.index)

In [38]:
# text_length / word_count
df['text_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

In [39]:
# downcast for memory efficiency
for col in ['turn_count', 'response_gap_mins', 'user_message_count', 'text_length', 'word_count']:
    df[col] = pd.to_numeric(df[col], downcast='integer')

In [40]:
#release date list with versioning
release_dates = pd.to_datetime([
    '2004-10-26',  # 4.10 Warty Warthog
    '2005-04-08',  # 5.04 Hoary Hedgehog
    '2005-10-12',  # 5.10 Breezy Badger
    '2006-06-01',  # 6.06 LTS Dapper Drake
    '2006-10-26',  # 6.10 Edgy Eft
    '2007-04-19',  # 7.04 Feisty Fawn
    '2007-10-18',  # 7.10 Gutsy Gibbon
    '2008-04-24',  # 8.04 LTS Hardy Heron
    '2008-10-30',  # 8.10 Intrepid Ibex
    '2009-04-23',  # 9.04 Jaunty Jackalope
    '2009-10-29',  # 9.10 Karmic Koala
    '2010-04-29',  # 10.04 LTS Lucid Lynx
    '2010-10-10',  # 10.10 Maverick Meerkat
    '2011-04-28',  # 11.04 Natty Narwhal
    '2011-10-13',  # 11.10 Oneiric Ocelot
    '2012-04-26',  # 12.04 LTS Precise Pangolin
    '2012-10-18',  # 12.10 Quantal Quetzal
    '2013-04-25',  # 13.04 Raring Ringtail
    '2013-10-17',  # 13.10 Saucy Salamander
    '2014-04-17',  # 14.04 LTS Trusty Tahr
    '2014-10-23',  # 14.10 Utopic Unicorn
    '2015-04-23',  # 15.04 Vivid Vervet
    '2015-10-22',  # 15.10 Wily Werewolf
    '2016-04-21',  # 16.04 LTS Xenial Xerus
]).sort_values()

In [41]:
#days_since_release / days_until_release
df = df.sort_values('date')
df = pd.merge_asof(df, pd.DataFrame({'date': release_dates, 'prior_release': release_dates}), on='date', direction='backward')
df = pd.merge_asof(df, pd.DataFrame({'date': release_dates, 'next_release': release_dates}), on='date', direction='forward')

df['days_since_release'] = (df['date'] - df['prior_release']).dt.days
df['days_until_release'] = (df['next_release'] - df['date']).dt.days

df = df.drop(columns=['prior_release', 'next_release'])
df = df.sort_values(['conversation_id', 'date']).reset_index(drop=True)

In [42]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8589731 entries, 0 to 8589730
Data columns (total 13 columns):
 #   Column              Dtype         
---  ------              -----         
 0   date                datetime64[us]
 1   from                category      
 2   to                  category      
 3   text                str           
 4   conversation_id     uint32        
 5   turn_count          int16         
 6   is_op               bool          
 7   response_gap_mins   float64       
 8   user_message_count  int32         
 9   text_length         int16         
 10  word_count          int16         
 11  days_since_release  float64       
 12  days_until_release  int64         
dtypes: bool(1), category(2), datetime64[us](1), float64(2), int16(3), int32(1), int64(1), str(1), uint32(1)
memory usage: 982.8 MB


In [43]:
df.sample(1)

,date,from,to,text,conversation_id,turn_count,is_op,response_gap_mins,user_message_count,text_length,word_count,days_since_release,days_until_release
963496,2006-03-15 14:49:00,Seveas,marcel,stop repeating,335409,1,False,0.0,18653,14,2,154.0,77


In [44]:
df[df['days_since_release'].isna()].sort_values('date')[['date', 'from', 'to', 'text']].head(10)

,date,from,to,text
8570182,2004-07-05 12:45:00,fabbione,mdz,"i think we could import the old comments via rsync, but from there we need to go via email. I think it is easier than caching the status on each bug and than import bits here and there"
8570183,2004-07-05 12:46:00,mdz,fabbione,it would be very easy to keep a hash db of message-ids
8570185,2004-07-05 12:50:00,fabbione,mdz,ok
8570184,2004-07-05 12:50:00,mdz,fabbione,sounds good
3485465,2004-08-17 07:19:00,rburton,__ALL__,the X keyboard layout debconf thing should give gb as an example for england as i can never remember what the code is
3485466,2004-08-17 07:20:00,Keybuk,rburton,that's going to die MUAHAHAHAHAHAHAHAHAHAHAHAHAHA!*cough*splutter*
6076217,2004-08-18 05:32:00,bob2,seb128,gnome-settings-daemon is working!
6076218,2004-08-18 05:33:00,seb128,bob2,Kamion has fixed it
7864784,2004-08-18 07:48:00,seb128,Kamion,do you know about this problem ?
7864785,2004-08-18 07:48:00,Kamion,seb128,no


NOTE: Ubuntu's IRC channels began on July 5, 2004. The earliest chats in this dataset are from that exact day, explaining the many chats with NaN values for days_since_release...as users were posting on the forums before the actual first release on October 26, 2004.

Additionally, as mentioned earlier, the response_gap_mins will always be N/A when turn count = 0

In [45]:
df.isna().sum()

date                        0
from                        0
to                          0
text                        0
conversation_id             0
turn_count                  0
is_op                       0
response_gap_mins     1008359
user_message_count          0
text_length                 0
word_count                  0
days_since_release      38524
days_until_release          0
dtype: int64

In [46]:
# Step 6 checkpoint — hand off the fully processed dataframe to the Steps 7-12 notebook
df.to_pickle('ubuntu_dialogue_step6.pkl')
print(f"Saved {len(df):,} rows, {len(df.columns)} columns -> ubuntu_dialogue_step6.pkl")
print(df.dtypes)

Saved 8,589,731 rows, 13 columns -> ubuntu_dialogue_step6.pkl
date                  datetime64[us]
from                        category
to                          category
text                             str
conversation_id               uint32
turn_count                     int16
is_op                           bool
response_gap_mins            float64
user_message_count             int32
text_length                    int16
word_count                     int16
days_since_release           float64
days_until_release             int64
dtype: object
